# Apt 305 — the AU corrections on a **closed** energy balance

Runs the seven-state harness end to end and checks the hard gate:

> **V2 Sankey closure residual < 5 % on the canonical baseline and every
> downstream state.** Until that passes, no reduction percentage and no
> kWh/m² headline from this model is a result.

What is being fixed, and why the earlier numbers could not be trusted:

| | |
| --- | --- |
| **Defect A** | The five party surfaces — 75.10 m², **88.6 % of the envelope UA** — were in the solver but in neither side of the reported balance. The Sankey listed two transmission entries for a building with seven surfaces. |
| **also A** | Transmission was read at each element's *internal* face while the gains were counted in full, so the radiative fractions ISO 52016-1 formula (39) deposits straight onto the surface nodes were invisible to it — 1 084.58 kWh, exactly. |
| **Defect B** | Latent cooling was charged in 8 758 of 8 760 hours against ~146 hours of actual plant operation. A moisture balance reported as plant energy, inflating the headline ~5×. |
| **Upstream** | `sky_view_factor == 0` was read as *slab-on-ground*, burying a third-floor apartment's ceiling in the earth and producing two irreconcilable baselines from one building. |

**Runtime: roughly 5–10 minutes.** Seven full annual simulations, one engine
worktree per state.

Runtime type: CPU is fine — there is nothing here a GPU would help with.

## 1 · Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO   = Path('/content/AIB')
BRANCH = 'claude/aib-energy-balance-closure-83epu7'   # the closed-balance branch

if not REPO.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/samiraghafarigousheh-sys/aib.git', str(REPO)],
                   check=True)
os.chdir(REPO)

# Every branch, not just the default one: the harness builds one worktree per
# state and each state IS a branch.
subprocess.run(['git', 'fetch', 'origin', '+refs/heads/*:refs/remotes/origin/*'],
               check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)

# Colab already ships numpy, pandas, matplotlib, plotly, tqdm and pytest.
# These four are the gaps.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pvlib', 'timezonefinder', 'holidays', 'workalendar'], check=True)

print(subprocess.run(['git', 'log', '--oneline', '-5'],
                     capture_output=True, text=True).stdout)
print('weather:', *[p.name for p in Path('weather_cache').glob('*.epw')])

## 2 · Run the harness

Seven states. For each one the engine branch is checked out into its own
worktree and the three **closure commits are cherry-picked on top**, so the
reporting instrument is identical across states while the physics fix under
test varies. Watch the `resid=` column: it is the whole point of the run.

In [ ]:
EPW = 'weather_cache/AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025.epw'

proc = subprocess.Popen(
    [sys.executable, 'tools/diagnostics/closed_balance_six_state.py',
     '--weather', EPW, '--outdir', 'results/au_corrections_closed'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:          # stream, rather than going quiet for ten minutes
    print(line, end='')
proc.wait()

print('\nexit code:', proc.returncode,
      '— 0 = gate passed on every state, 2 = at least one state failed')

## 3 · The gate

In [ ]:
import json
import pandas as pd

RAW = Path('results/au_corrections_closed/six_state_closed_raw.json')
blob = json.loads(RAW.read_text())
results, meta = blob['results'], blob['meta']

# What the same column read before these fixes, for contrast.
BEFORE = {'Baseline': 62.41, '+Vent+Latent': 58.61, '+Internal Gains': 52.06,
          '+Conditioned Zones': -22.52, '+Ground Fix': -19.64, '+Hemisphere Fix': -19.64}

rows = []
for state, r in results.items():
    b, sk = r['config_B'], r['config_B']['sankey']
    rows.append({
        'State': state,
        'Inputs (kWh)':  sk['inputs_total_Wh'] / 1000,
        'Outputs (kWh)': sk['outputs_total_Wh'] / 1000,
        'Residual (kWh)': sk['residual_kWh'],
        'Residual %': sk['residual_pct'],
        'Was %': BEFORE.get(state, float('nan')),
        'Transmission items': sk['n_transmission_items'],
        'Gate': 'PASS' if abs(sk['residual_pct']) < 5.0 else 'FAIL',
    })

gate = pd.DataFrame(rows)
display(gate.style.format({
    'Inputs (kWh)': '{:,.2f}', 'Outputs (kWh)': '{:,.2f}',
    'Residual (kWh)': '{:+,.2f}', 'Residual %': '{:+.2f} %', 'Was %': '{:+.2f} %',
}, na_rep='— (new state)').hide(axis='index'))

assert (gate['Gate'] == 'PASS').all(), 'GATE FAILED — do not quote a headline from this run'
print('\nGate passed on all', len(gate), 'states.')
print('Every state now lists', gate['Transmission items'].iloc[0],
      'transmission line items; before the fix it was 2, for a building with 7 surfaces.')

### What the residual that survives is made of

On the four states *before* the ground fix the residual is not noise. It is the
phantom ground term — a lumped `h_ground · (T_in − T_gr)` flow computed from a
slab area that `_ground_contact_area()` filled in from `net_floor_area` because
no surface carried a ground tag, with no `GR` element behind it in the solver.

In [ ]:
comp = pd.DataFrame([{
    'State': s,
    'Residual (kWh)': r['config_B']['sankey']['residual_kWh'],
    '−(ground loss − gain) (kWh)': -((r['config_B'].get('Q_ground_loss_kWh') or 0.0)
                                     - (r['config_B'].get('Q_ground_gain_kWh') or 0.0)),
} for s, r in results.items()])
comp['|Δ| (kWh)'] = (comp['Residual (kWh)'] - comp['−(ground loss − gain) (kWh)']).abs()

display(comp.style.format({
    'Residual (kWh)': '{:+,.2f}', '−(ground loss − gain) (kWh)': '{:+,.2f}',
    '|Δ| (kWh)': '{:.2e}'}).hide(axis='index'))

print('They match to within', f"{comp['|Δ| (kWh)'].max():.1e}", 'kWh — the same term, twice.')
print('The ground fix removes it and the residual goes to zero: the +66.61 kWh')
print('effect the Item 3 diagnosis attributed to it, now measured on a balance')
print('where it is the only thing left to remove.')

## 4 · Energy need, sensible and latent kept apart

In [ ]:
need = pd.DataFrame([{
    'State': s,
    'Sensible heating (kWh)': r['config_B']['Q_H_sensible_kWh'],
    'Sensible cooling (kWh)': r['config_B']['Q_C_sensible_kWh'],
    'Latent cooling, gated (kWh)': r['config_B']['Q_C_latent_kWh'],
    'Latent cooling, ungated (kWh)': r['config_B']['Q_C_latent_ungated_kWh'],
    'Latent heating (kWh)': r['config_B']['Q_H_latent_kWh'],
    'Total (kWh)': r['config_B']['Q_need_total_kWh'],
    'Total (kWh/m²)': r['config_B']['Q_need_total_kWh_per_sqm'],
} for s, r in results.items()])

display(need.style.format({c: '{:,.2f}' for c in need.columns if c != 'State'}
                          | {'Latent heating (kWh)': '{:,.4f}'}).hide(axis='index'))

head = need.iloc[-1]
print(f"Final state: {head['Sensible heating (kWh)']:.2f} + {head['Sensible cooling (kWh)']:.2f} "
      f"sensible + {head['Latent cooling, gated (kWh)']:.2f} latent "
      f"= {head['Total (kWh)']:.2f} kWh ({head['Total (kWh/m²)']:.2f} kWh/m²·yr)")
print("The ungated definition reported 697.10 kWh / 34.85 kWh/m² for the same run.")

### The latent gate

Latent cooling is charged only where **both** hold: the cooling plant is running
(`Q_C > 0`) and the moisture balance actually calls for dehumidification. The two
columns that have to be zero are the last two.

In [ ]:
lat = pd.DataFrame([{
    'State': s,
    'Steps': r['config_B'].get('n_steps'),
    'Cooling plant on': r['config_B'].get('n_steps_cooling_on'),
    'Latent charged': r['config_B'].get('n_steps_latent_charged'),
    'Gated (kWh)': r['config_B']['Q_C_latent_kWh'],
    'Ungated (kWh)': r['config_B']['Q_C_latent_ungated_kWh'],
    'Charged w/ cooling OFF': r['config_B'].get('latent_kWh_with_cooling_off'),
    'Charged while HEATING': r['config_B'].get('latent_kWh_while_heating'),
} for s, r in results.items()])
display(lat.style.format({'Gated (kWh)': '{:,.2f}', 'Ungated (kWh)': '{:,.2f}',
                          'Charged w/ cooling OFF': '{:.4f}',
                          'Charged while HEATING': '{:.4f}'}).hide(axis='index'))

assert lat['Charged w/ cooling OFF'].abs().max() < 1e-9
assert lat['Charged while HEATING'].abs().max() < 1e-9
print('Zero with the plant off, zero while the heating plant runs — in every state.')

# The gate must not have inverted the seasonal phase.
import calendar
monthly = results[list(results)[-1]]['config_B'].get('monthly_latent_cooling_kWh')
if monthly:
    peak = max(range(12), key=lambda i: monthly[i]) + 1
    print(f'Monthly latent cooling peaks in {calendar.month_name[peak]} '
          f'— southern-hemisphere phase retained.')

## 5 · The adjacent-zone surfaces are in the inventory

75.10 m², 88.6 % of the envelope UA, previously absent from both sides. The last
column is the consistency check the plan asked for: the Sankey's transmission
line items against an **independent** re-integration of the per-surface flows
from the hourly frame — two different aggregation paths, tolerance 0.1 %.

In [ ]:
adj = pd.DataFrame([{
    'State': s,
    'ADJ loss (kWh)': r['config_B'].get('Q_tr_adjacent_loss_kWh'),
    'ADJ gain (kWh)': r['config_B'].get('Q_tr_adjacent_gain_kWh'),
    'Line items': r['config_B']['sankey']['n_transmission_items'],
    'Reported Σ (kWh)': r['config_B']['sankey']['transmission_reported_kWh'],
    'Independent Σ (kWh)': r['config_B']['sankey']['transmission_independent_kWh'],
    'Δ %': 100 * r['config_B']['sankey']['transmission_rel_diff'],
} for s, r in results.items()])
display(adj.style.format({'ADJ loss (kWh)': '{:,.2f}', 'ADJ gain (kWh)': '{:,.2f}',
                          'Reported Σ (kWh)': '{:,.2f}', 'Independent Σ (kWh)': '{:,.2f}',
                          'Δ %': '{:.4f} %'}).hide(axis='index'))

# The per-surface inventory of the final state, in full.
print('\nFinal-state transmission line items (kWh):')
for name, kwh in sorted(results[list(results)[-1]]['config_B']['sankey']
                        ['transmission_items_kWh'].items(), key=lambda kv: -kv[1]):
    print(f'  {name:<58} {kwh:9.2f}')

print('\nISO 52016-1 element class per surface:')
for name, t in results[list(results)[-1]]['config_B']['sankey']['surface_iso_types'].items():
    print(f'  {name:<58} {t}')

## 6 · Config A / config B convergence

Config A types the five party surfaces `"opaque"`; config B types them
`"adjacent"`. Before the classification fix, config A buried all five as
slab-on-ground and returned **15.86 kWh heating / 2 027.5 kWh cooling** against
config B's 1 308.60 / 741.83 — one building, two irreconcilable baselines.

In [ ]:
conv_state = next((s for s, r in results.items() if 'config_A' in r), None)
if conv_state is None:
    print('no config-A run in this output')
else:
    A, B = results[conv_state]['config_A'], results[conv_state]['config_B']
    keys = ['Q_H_sensible_kWh', 'Q_C_sensible_kWh', 'Q_need_total_kWh',
            'Q_tr_adjacent_loss_kWh', 'Q_ground_loss_kWh']
    conv = pd.DataFrame([{'Metric': k, 'Config A': A[k], 'Config B': B[k],
                          'Difference': abs(A[k] - B[k])} for k in keys])
    display(conv.style.format({'Config A': '{:,.6f}', 'Config B': '{:,.6f}',
                               'Difference': '{:.3e}'}).hide(axis='index'))

    gr_a = [n for n, t in A['sankey']['surface_iso_types'].items() if t == 'GR']
    gr_b = [n for n, t in B['sankey']['surface_iso_types'].items() if t == 'GR']
    print(f'Surfaces classified GR — config A: {gr_a or "none"}, config B: {gr_b or "none"}')
    print('Converged bit-exactly.' if conv['Difference'].max() == 0
          else f"Largest difference: {conv['Difference'].max():.3e} kWh")

## 7 · The chart

In [ ]:
subprocess.run([sys.executable, 'tools/diagnostics/make_closed_balance_chart.py'], check=True)

from IPython.display import Image, display
display(Image('results/au_corrections_closed/au_corrections_closed_balance.png'))

## 8 · The regression tests

These are the assertions behind the tables above — the ADJ surfaces present as
line items, the 0.1 % consistency check, the gate, the latent gating, the
classifier, and config A/B convergence. Plus a ground-floor building, so the
ADJ fix cannot be a blanket "ground is always zero".

In [ ]:
env = dict(os.environ, PYTHONPATH=str(REPO / 'pybuildingenergy' / 'src'))
subprocess.run([sys.executable, '-m', 'pytest',
                'tests/test_sankey_closure_adj_transmission.py',
                'tests/test_latent_gating.py',
                'tests/test_gr_classification.py',
                '-v', '--no-header', '-p', 'no:cacheprovider'], env=env)

## 9 · The written report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/au_corrections_closed/six_state_closed.md').read_text()))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('results/au_corrections_closed/NOTES.md').read_text()))

## 10 · Before quoting any of this

Four things that will otherwise be got wrong:

1. **`+Hemisphere Fix` and `+Closure Fixes (HEAD)` are different engines.** The
   seventh row is `main` plus the three closure commits; the sixth is the
   historical branch. They differ by what landed on `main` afterwards — chiefly
   the dynamic external convective coefficient, which moves sensible cooling
   20.06 → 67.12 kWh. Quote one, say which, and do not average them.

2. **The six historical states are measured with a back-ported instrument.** The
   closure commits are cherry-picked onto each branch so the reporting is
   identical across states. The *physics* of each state is untouched. The
   mechanics — one compatibility alias, one conflict resolved toward the branch —
   are in §6 of the report above and in the harness docstring. Anything
   conflicting outside `check_input.py` aborts the run rather than being guessed.

3. **`Q_tr_*` means something different now.** Outer-face and net of absorbed
   short-wave (sol-air), not internal-face. That is what closes the balance, but
   it means these numbers will not match anything published from the old
   columns.

4. **The old −79.8 % headline came off an unclosed balance** — a residual running
   62 % → −20 %, with 88.6 % of the envelope missing from the inventory. Recompute
   any reduction from the table in §4, not from the earlier summaries.

The gate having passed, the methodology text can now be written against these
numbers.